In [1]:
print ("fhello")


fhello


In [2]:
import cv2
import numpy as np

# Open USB camera (0 = default webcam)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Cannot access camera")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize for speed
    frame = cv2.resize(frame, (640, 480))

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Edge detection
    edges = cv2.Canny(blurred, 50, 150)

    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Ignore small noise
        if area < 1000:
            continue

        # Approximate contour
        epsilon = 0.02 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        # If rectangle (4 corners)
        if len(approx) == 4:
            x, y, w, h = cv2.boundingRect(approx)

            # Draw rectangle
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

            cv2.putText(frame, "Block",
                        (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2)

    # Show output
    cv2.imshow("Block Detection", frame)
    cv2.imshow("Edges", edges)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from ultralytics import YOLO
import cv2

# Load YOLOv8 nano model (fastest)
model = YOLO("yolov8n.pt")

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Cannot open camera")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO detection
    results = model(frame, conf=0.5)  # confidence threshold

    # Draw results on frame
    annotated_frame = results[0].plot()

    # Show output
    cv2.imshow("YOLO Detection", annotated_frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [3]:
import cv2

def list_cameras(max_devices=10):
    available = []
    for i in range(max_devices):
        cap = cv2.VideoCapture(i)
        if cap is not None and cap.isOpened():
            available.append(i)
            cap.release()
    return available

print("Available camera indices:", list_cameras())

Available camera indices: [0]


In [4]:
import cv2
import numpy as np
import time

# ---------------- CAMERA ----------------
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

if not cap.isOpened():
    print("Error: Cannot open camera")
    exit()

# ---------------- COLOR RANGES (HSV) ----------------
# Tune these depending on your lighting
color_ranges = {
    "red1": ([0, 120, 70], [10, 255, 255]),
    "red2": ([170, 120, 70], [180, 255, 255]),  # red wraparound
    "blue": ([100, 150, 50], [140, 255, 255]),
    "green": ([40, 70, 70], [80, 255, 255]),
    "yellow": ([20, 100, 100], [35, 255, 255])
}

# ---------------- PARAMETERS ----------------
MIN_AREA = 1500
ASPECT_RATIO_TOL = 0.4   # how square the block should be

kernel = np.ones((5, 5), np.uint8)

prev_time = time.time()

# ---------------- MAIN LOOP ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)  # mirror (optional)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    detected_blocks = []

    # ---------- PROCESS EACH COLOR ----------
    for color_name, (lower, upper) in color_ranges.items():
        lower = np.array(lower)
        upper = np.array(upper)

        mask = cv2.inRange(hsv, lower, upper)

        # Combine red ranges
        if color_name == "red1":
            lower2, upper2 = color_ranges["red2"]
            mask2 = cv2.inRange(hsv, np.array(lower2), np.array(upper2))
            mask = mask | mask2

        # ---------- CLEAN MASK ----------
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # ---------- FIND BLOCKS ----------
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < MIN_AREA:
                continue

            rect = cv2.minAreaRect(cnt)
            (cx, cy), (w, h), angle = rect

            if w == 0 or h == 0:
                continue

            # ---------- FILTER BY SHAPE (square-like) ----------
            aspect_ratio = min(w, h) / max(w, h)

            if aspect_ratio < (1 - ASPECT_RATIO_TOL):
                continue

            cx, cy = int(cx), int(cy)

            # Normalize angle
            if w < h:
                angle = angle + 90

            # ---------- DRAW ----------
            box = cv2.boxPoints(rect)
            box = np.intp(box)

            cv2.drawContours(frame, [box], 0, (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

            label = f"{color_name} ({cx},{cy}) {angle:.1f}"
            cv2.putText(frame, label,
                        (cx - 60, cy - 15),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 255), 2)

            detected_blocks.append((color_name, cx, cy, angle))

    # ---------- FPS DISPLAY ----------
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time

    cv2.putText(frame, f"FPS: {int(fps)}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255), 2)

    # ---------- PRINT DATA ----------
    for block in detected_blocks:
        print(f"Detected: {block}")

    # ---------- SHOW ----------
    cv2.imshow("Advanced Block Detection", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        break

# ---------------- CLEANUP ----------------
cap.release()
cv2.destroyAllWindows()

In [6]:
import cv2
import numpy as np
import time

# ---------------- CAMERA ----------------
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

if not cap.isOpened():
    print("Error: Cannot open camera")
    exit()

# ---------------- COLOR RANGES (HSV) ----------------
color_ranges = {
    "red1": ([0, 120, 70], [10, 255, 255]),
    "red2": ([170, 120, 70], [180, 255, 255]),
    "blue": ([100, 150, 50], [140, 255, 255]),
    "green": ([40, 70, 70], [80, 255, 255]),
    "yellow": ([20, 100, 100], [35, 255, 255])
}

# ---------------- PARAMETERS ----------------
MIN_AREA = 1500
ASPECT_RATIO_TOL = 0.4
SMOOTHING = 0.7

kernel = np.ones((5, 5), np.uint8)

prev_centers = {}
locked_target = None

prev_time = time.time()

# ---------------- MAIN LOOP ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    detected_blocks = []

    # ---------- DETECTION ----------
    for color_name, (lower, upper) in color_ranges.items():
        if color_name == "red2":
            continue

        lower = np.array(lower)
        upper = np.array(upper)

        mask = cv2.inRange(hsv, lower, upper)

        # Combine red ranges
        if color_name == "red1":
            lower2, upper2 = color_ranges["red2"]
            mask2 = cv2.inRange(hsv, np.array(lower2), np.array(upper2))
            mask = mask | mask2
            color_name = "red"

        # Clean mask
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < MIN_AREA:
                continue

            rect = cv2.minAreaRect(cnt)
            (cx, cy), (w, h), angle = rect

            if w == 0 or h == 0:
                continue

            # Square filter
            aspect_ratio = min(w, h) / max(w, h)
            if aspect_ratio < (1 - ASPECT_RATIO_TOL):
                continue

            cx, cy = int(cx), int(cy)

            # Normalize angle
            if w < h:
                angle += 90

            # ---------- SMOOTHING ----------
            key = color_name
            if key in prev_centers:
                px, py = prev_centers[key]
                cx = int(SMOOTHING * px + (1 - SMOOTHING) * cx)
                cy = int(SMOOTHING * py + (1 - SMOOTHING) * cy)

            prev_centers[key] = (cx, cy)

            detected_blocks.append((color_name, cx, cy, angle, rect))

    # ---------- TARGET SELECTION ----------
    best_block = None
    frame_center = (320, 240)
    min_dist = float('inf')

    for block in detected_blocks:
        color, cx, cy, angle, rect = block
        dist = (cx - frame_center[0])**2 + (cy - frame_center[1])**2

        if dist < min_dist:
            min_dist = dist
            best_block = block

    # ---------- LOCK-ON MECHANISM ----------
    if locked_target is None and best_block:
        locked_target = best_block

    # If locked, try to keep same color
    if locked_target:
        target_color = locked_target[0]
        candidates = [b for b in detected_blocks if b[0] == target_color]

        if candidates:
            # choose closest among same color
            best_block = min(
                candidates,
                key=lambda b: (b[1]-frame_center[0])**2 + (b[2]-frame_center[1])**2
            )
            locked_target = best_block
        else:
            locked_target = None

    # ---------- DRAW + OUTPUT ----------
    if locked_target:
        color, cx, cy, angle, rect = locked_target

        box = cv2.boxPoints(rect)
        box = np.intp(box)

        cv2.drawContours(frame, [box], 0, (255, 0, 0), 3)
        cv2.circle(frame, (cx, cy), 8, (0, 0, 255), -1)

        # Normalized coordinates
        nx = (cx - 320) / 320
        ny = (cy - 240) / 240

        label = f"{color} TARGET"
        cv2.putText(frame, label, (cx - 60, cy - 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

        # Print robot-ready data
        print(f"{color},{nx:.2f},{ny:.2f},{angle:.2f}")

    # ---------- FPS ----------
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time

    cv2.putText(frame, f"FPS: {int(fps)}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255), 2)

    # ---------- DISPLAY ----------
    cv2.imshow("FINAL Block Detection System", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        break

# ---------------- CLEANUP ----------------
cap.release()
cv2.destroyAllWindows()

red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
red,0.85,0.78,-90.00
